In [ ]:
%load_ext autoreload
%autoreload 2

# Module D — Ephys Convergence: Cross-Species Harmonization

This notebook validates that human CGE interneuron cluster identities (from
scVI/scANVI mapping) are supported by electrophysiological evidence.
It aligns mouse (DANDI 000008) and human (DANDI 000636) patch-seq cells in
a shared ephys feature space after within-species z-scoring, then tests
whether the cross-species cluster correspondence is greater than chance.

## Tier 1 — Parametric harmonization (this notebook)

1. Load combined ephys features (generated by `scripts/10_aggregate_ephys.py`)
2. Attach cluster assignments from transcriptomic mapping
3. Within-species z-scoring of all features
4. Global permutation test for cross-species cluster similarity
5. Per-cluster comparison (box plots + Euclidean distance heatmap)
6. CCKBC-specific analysis

## Tier 2 — ComBat batch correction (TODO)

See TODO comment below.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from pathlib import Path
import sys

matplotlib.rcParams["figure.facecolor"] = "none"
matplotlib.rcParams["axes.facecolor"] = "none"
matplotlib.rcParams["savefig.transparent"] = True

# Project imports
_REPO_ROOT = Path("../..").resolve()
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from cge_subtype.src.ephys_harmonization import (
    BIO_FEATURES,
    ISI_FEATURES,
    CV_ISI_FEATURES,
    ALL_FEATURES,
    LOG_FEATURES,
    zscore_within_species,
    permutation_test_cluster_similarity,
)

In [ ]:
# Configuration
OUTDIR = Path("../../results/ephys_convergence")
OUTDIR.mkdir(parents=True, exist_ok=True)

FIGURE_DPI = 150
SAVE_FIGURES = True

## 1. Load Aggregated Ephys Features

**TODO**: Run `scripts/10_aggregate_ephys.py` first to generate these files.
Expected paths:
  - `results/ephys_convergence/combined_ephys_features.csv`
  - `results/ephys_convergence/mouse_ephys_features.csv`
  - `results/ephys_convergence/human_ephys_features.csv`

In [ ]:
COMBINED_EPHYS_PATH = OUTDIR / "combined_ephys_features.csv"
MOUSE_EPHYS_PATH = OUTDIR / "mouse_ephys_features.csv"
HUMAN_EPHYS_PATH = OUTDIR / "human_ephys_features.csv"

if COMBINED_EPHYS_PATH.exists():
    combined_df = pd.read_csv(COMBINED_EPHYS_PATH)
    mouse_df = pd.read_csv(MOUSE_EPHYS_PATH)
    human_df = pd.read_csv(HUMAN_EPHYS_PATH)
    print(f"Combined: {combined_df.shape}")
    print(f"  Mouse: {(combined_df['species'] == 'mouse').sum()} cells")
    print(f"  Human: {(combined_df['species'] == 'human').sum()} cells")
else:
    # Placeholder: create synthetic data for notebook development
    print(f"WARNING: {COMBINED_EPHYS_PATH} not found.")
    print("Using synthetic placeholder data.")
    rng = np.random.default_rng(42)
    n_mouse, n_human = 200, 150
    feat_cols = BIO_FEATURES + ISI_FEATURES[:6]
    mouse_synthetic = pd.DataFrame(
        rng.standard_normal((n_mouse, len(feat_cols))),
        columns=feat_cols,
    )
    mouse_synthetic["cell_id"] = [f"mouse_{i}" for i in range(n_mouse)]
    mouse_synthetic["species"] = "mouse"
    human_synthetic = pd.DataFrame(
        rng.standard_normal((n_human, len(feat_cols))),
        columns=feat_cols,
    )
    human_synthetic["cell_id"] = [f"human_{i}" for i in range(n_human)]
    human_synthetic["species"] = "human"
    mouse_df = mouse_synthetic
    human_df = human_synthetic
    combined_df = pd.concat([mouse_df, human_df], ignore_index=True)
    print(f"Synthetic combined shape: {combined_df.shape}")

## 2. Load Cluster Assignments

**TODO**: Load cluster assignments from transcriptomic mapping results.
Expected paths:
  - Mouse clusters: `results/mouse_patchseq_cluster_assignments.csv`
    (columns: cell_id, cluster_label, t-type)
  - Human clusters: `results/human_scvi_mapping_results.csv`
    (columns: cell_id, predicted_cluster, confidence)
  - Cross-species mapping: `results/cross_species_rbh_mapping.csv`
    (columns: mouse_cluster, human_cluster, correlation, is_rbh)

In [ ]:
MOUSE_CLUSTER_PATH = _REPO_ROOT / "cge_subtype/results/mouse_patchseq_cluster_assignments.csv"
HUMAN_CLUSTER_PATH = _REPO_ROOT / "cge_subtype/results/human_scvi_mapping_results.csv"
CROSS_SPECIES_MAP_PATH = _REPO_ROOT / "cge_subtype/results/cross_species_rbh_mapping.csv"

if MOUSE_CLUSTER_PATH.exists() and HUMAN_CLUSTER_PATH.exists():
    mouse_clusters = pd.read_csv(MOUSE_CLUSTER_PATH)
    human_clusters = pd.read_csv(HUMAN_CLUSTER_PATH)
    print(f"Mouse clusters: {mouse_clusters.shape}")
    print(f"Human clusters: {human_clusters.shape}")
else:
    # TODO: Replace once transcriptomic mapping results are available
    print(f"WARNING: Cluster assignment files not found.")
    print("Using synthetic cluster labels for development.")
    rng = np.random.default_rng(0)
    n_mouse = len(mouse_df)
    n_human = len(human_df)
    mouse_clusters = pd.DataFrame({
        "cell_id": mouse_df["cell_id"].values,
        "cluster_label": rng.choice(["C0", "C1", "C2", "CCKBC"], size=n_mouse),
    })
    human_clusters = pd.DataFrame({
        "cell_id": human_df["cell_id"].values,
        "predicted_cluster": rng.choice(["C0", "C1", "C2", "CCKBC"], size=n_human),
    })

In [ ]:
# Merge cluster labels into combined_df
mouse_merged = combined_df[combined_df["species"] == "mouse"].merge(
    mouse_clusters.rename(columns={"cluster_label": "cluster"})[["cell_id", "cluster"]],
    on="cell_id",
    how="left",
)
human_merged = combined_df[combined_df["species"] == "human"].merge(
    human_clusters.rename(columns={"predicted_cluster": "cluster"})[["cell_id", "cluster"]],
    on="cell_id",
    how="left",
)
combined_annotated = pd.concat([mouse_merged, human_merged], ignore_index=True)
print(f"Annotated combined: {combined_annotated.shape}")
print(f"Cluster NaN rate: {combined_annotated['cluster'].isna().mean():.1%}")

## 3. Tier 1: Within-Species Z-Scoring

In [ ]:
# Select numeric feature columns present in the data
feat_cols = [c for c in BIO_FEATURES if c in combined_annotated.columns]
isi_cols = [c for c in ISI_FEATURES if c in combined_annotated.columns]
cv_cols = [c for c in CV_ISI_FEATURES if c in combined_annotated.columns]
analysis_cols = feat_cols + isi_cols + cv_cols

print(f"Feature columns available: {len(analysis_cols)}")
print(f"  BIO: {len(feat_cols)}, ISI: {len(isi_cols)}, CV_ISI: {len(cv_cols)}")

# Drop rows missing all features (e.g. silent cells)
feat_present = combined_annotated[analysis_cols].notna().any(axis=1)
combined_valid = combined_annotated[feat_present].copy().reset_index(drop=True)
print(f"Cells with at least one feature: {len(combined_valid)} / {len(combined_annotated)}")

In [ ]:
# Z-score within species
features_raw = combined_valid[analysis_cols]
species_labels = combined_valid["species"].values
features_zscored = zscore_within_species(features_raw, species_labels)

print("Z-scored features shape:", features_zscored.shape)
# Sanity check: mean per species should be ~0
for sp in ["mouse", "human"]:
    mask = species_labels == sp
    sp_mean = features_zscored.loc[mask].mean().mean()
    print(f"  {sp} grand mean: {sp_mean:.4f} (expect ~0)")

## 4. Global Permutation Test

Test whether clusters with cells from both species have more similar
centroids than expected by chance (shuffled cluster labels).

In [ ]:
# Drop cells without cluster assignment for the permutation test
has_cluster = combined_valid["cluster"].notna()
perm_features = features_zscored[has_cluster]
perm_clusters = combined_valid.loc[has_cluster, "cluster"].values
perm_species = combined_valid.loc[has_cluster, "species"].values

print(f"Cells for permutation test: {len(perm_features)}")
print(f"Clusters: {np.unique(perm_clusters)}")

In [ ]:
N_PERM = 1000
SEED = 42

print(f"Running permutation test (n_perm={N_PERM}) …")
p_value = permutation_test_cluster_similarity(
    perm_features,
    perm_clusters,
    perm_species,
    n_perm=N_PERM,
    seed=SEED,
)
print(f"\nGlobal permutation test p-value: {p_value:.4f}")
if p_value < 0.05:
    print("=> Cross-species cluster similarity is GREATER than chance (p < 0.05)")
else:
    print("=> No significant cross-species cluster similarity detected")

## 5. Per-Cluster Comparison

In [ ]:
# Compute per-cluster, per-species centroids for BIO_FEATURES
centroid_data = []
for cl in np.unique(perm_clusters):
    for sp in ["mouse", "human"]:
        mask = (combined_valid["cluster"] == cl) & (combined_valid["species"] == sp)
        n_cells = mask.sum()
        if n_cells == 0:
            continue
        centroid = features_zscored.loc[mask, feat_cols].mean()
        centroid_data.append({
            "cluster": cl,
            "species": sp,
            "n_cells": n_cells,
            **centroid.to_dict(),
        })

centroids_df = pd.DataFrame(centroid_data)
print("Centroids computed for:")
print(centroids_df[["cluster", "species", "n_cells"]].to_string(index=False))

In [ ]:
# Euclidean distance between mouse/human centroids per cluster
dist_records = []
for cl in np.unique(perm_clusters):
    cl_centroids = centroids_df[centroids_df["cluster"] == cl]
    if not (
        (cl_centroids["species"] == "mouse").any()
        and (cl_centroids["species"] == "human").any()
    ):
        continue
    mouse_centroid = cl_centroids[cl_centroids["species"] == "mouse"][feat_cols].values[0]
    human_centroid = cl_centroids[cl_centroids["species"] == "human"][feat_cols].values[0]
    dist = np.sqrt(np.nansum((mouse_centroid - human_centroid) ** 2))
    dist_records.append({"cluster": cl, "cross_species_distance": dist})

dist_df = pd.DataFrame(dist_records).sort_values("cross_species_distance")
print("\nCross-species centroid distances (smaller = more similar):")
print(dist_df.to_string(index=False))

In [ ]:
# Box plots: key features by cluster and species
fig, axes = plt.subplots(1, min(4, len(feat_cols)), figsize=(16, 4))
axes = np.atleast_1d(axes)

key_features = [
    "spike_frequency_Hz",
    "avg_ap_width_ms",
    "avg_upstroke_downstroke_ratio",
    "avg_threshold_voltage_mV",
]
key_features = [f for f in key_features if f in features_zscored.columns]

cluster_colors = {"mouse": "#4C72B0", "human": "#DD8452"}

for ax, feat in zip(axes, key_features[:4]):
    data_by_group = []
    labels = []
    for cl in sorted(np.unique(perm_clusters)):
        for sp in ["mouse", "human"]:
            mask = (combined_valid["cluster"] == cl) & (combined_valid["species"] == sp)
            vals = features_zscored.loc[mask, feat].dropna().values
            if len(vals) == 0:
                continue
            data_by_group.append(vals)
            labels.append(f"{cl}\n({sp[:1]})")
    if data_by_group:
        bp = ax.boxplot(data_by_group, patch_artist=True, notch=False)
        for patch, lbl in zip(bp["boxes"], labels):
            sp_letter = lbl.split("(")[-1].rstrip(")")
            patch.set_facecolor(cluster_colors.get(
                "mouse" if sp_letter == "m" else "human", "#cccccc"
            ))
            patch.set_alpha(0.7)
        ax.set_xticks(range(1, len(labels) + 1))
        ax.set_xticklabels(labels, fontsize=7)
    ax.set_title(feat.replace("_", "\n"), fontsize=9)
    ax.set_ylabel("Z-score" if ax == axes[0] else "")
    ax.axhline(0, color="black", lw=0.5, linestyle="--", alpha=0.5)
    ax.set_facecolor("none")

# Legend
patches = [
    mpatches.Patch(color=cluster_colors["mouse"], label="Mouse"),
    mpatches.Patch(color=cluster_colors["human"], label="Human"),
]
axes[-1].legend(handles=patches, loc="upper right", fontsize=8)

fig.suptitle("Key Ephys Features by Cluster and Species (Z-scored within species)", fontsize=11)
fig.patch.set_alpha(0)
plt.tight_layout()

if SAVE_FIGURES:
    fig.savefig(OUTDIR / "cluster_ephys_boxplot.pdf", dpi=FIGURE_DPI, transparent=True)
    print(f"Saved: {OUTDIR / 'cluster_ephys_boxplot.pdf'}")
plt.show()

## 6. CCKBC-Specific Analysis

Compare CCKBC-assigned cells (from cluster mapping) between mouse and human.
CCK basket cells are expected to show: high spike frequency, narrow AP width,
low upstroke:downstroke ratio, fast trough after hyperpolarization.

In [ ]:
CCKBC_CLUSTER = "CCKBC"  # Update to match actual cluster label in your mapping

# CCKBC-relevant features
cckbc_features = [
    "spike_frequency_Hz",
    "avg_ap_width_ms",
    "avg_upstroke_downstroke_ratio",
    "avg_fast_trough_at_hyperpolarization_mV",
    "avg_threshold_voltage_mV",
    "current_threshold_pA",
]
cckbc_features = [f for f in cckbc_features if f in features_zscored.columns]

cckbc_mask = combined_valid["cluster"] == CCKBC_CLUSTER
n_cckbc_mouse = ((combined_valid["species"] == "mouse") & cckbc_mask).sum()
n_cckbc_human = ((combined_valid["species"] == "human") & cckbc_mask).sum()
print(f"CCKBC cells: {n_cckbc_mouse} mouse, {n_cckbc_human} human")

if n_cckbc_mouse > 0 and n_cckbc_human > 0:
    fig, axes = plt.subplots(1, len(cckbc_features), figsize=(3 * len(cckbc_features), 4))
    axes = np.atleast_1d(axes)

    from scipy import stats

    for ax, feat in zip(axes, cckbc_features):
        mouse_vals = features_zscored.loc[
            (combined_valid["species"] == "mouse") & cckbc_mask, feat
        ].dropna().values
        human_vals = features_zscored.loc[
            (combined_valid["species"] == "human") & cckbc_mask, feat
        ].dropna().values

        data_to_plot = []
        tick_labels = []
        colors = []
        for vals, sp in [(mouse_vals, "Mouse"), (human_vals, "Human")]:
            if len(vals) > 0:
                data_to_plot.append(vals)
                tick_labels.append(f"{sp}\n(n={len(vals)})")
                colors.append(cluster_colors[sp.lower()])

        if len(data_to_plot) == 2:
            bp = ax.boxplot(data_to_plot, patch_artist=True)
            for patch, col in zip(bp["boxes"], colors):
                patch.set_facecolor(col)
                patch.set_alpha(0.7)

            # Mann-Whitney U test
            if len(data_to_plot[0]) > 2 and len(data_to_plot[1]) > 2:
                stat, pval = stats.mannwhitneyu(
                    data_to_plot[0], data_to_plot[1], alternative="two-sided"
                )
                ax.set_title(f"{feat.replace('_', ' ')}\np={pval:.3f}", fontsize=8)
            else:
                ax.set_title(feat.replace("_", "\n"), fontsize=8)

            ax.set_xticks([1, 2])
            ax.set_xticklabels(tick_labels, fontsize=8)
            ax.axhline(0, color="black", lw=0.5, linestyle="--", alpha=0.5)
            ax.set_ylabel("Z-score" if ax == axes[0] else "")
        ax.set_facecolor("none")

    fig.suptitle("CCKBC Cluster: Mouse vs Human Ephys (Z-scored)", fontsize=11)
    fig.patch.set_alpha(0)
    plt.tight_layout()

    if SAVE_FIGURES:
        fig.savefig(OUTDIR / "cckbc_cross_species_ephys.pdf", dpi=FIGURE_DPI, transparent=True)
        print(f"Saved: {OUTDIR / 'cckbc_cross_species_ephys.pdf'}")
    plt.show()
else:
    print(f"Skipping CCKBC plot: insufficient cells in cluster '{CCKBC_CLUSTER}'.")
    print("TODO: Update CCKBC_CLUSTER to match actual cluster label in mapping results.")

## 7. PCA Visualization of Cross-Species Alignment

In [ ]:
# PCA on z-scored features (use BIO_FEATURES only — avoid sparse ISI bins)
pca_features = [f for f in feat_cols if f in features_zscored.columns]
pca_data = features_zscored[pca_features].dropna()
pca_idx = pca_data.index

if len(pca_data) > 10:
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(pca_data.values)
    coords_df = pd.DataFrame(coords, columns=["PC1", "PC2"], index=pca_idx)
    coords_df["species"] = combined_valid.loc[pca_idx, "species"].values
    coords_df["cluster"] = combined_valid.loc[pca_idx, "cluster"].values

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: colored by species
    for sp, col in [("mouse", "#4C72B0"), ("human", "#DD8452")]:
        mask = coords_df["species"] == sp
        axes[0].scatter(
            coords_df.loc[mask, "PC1"],
            coords_df.loc[mask, "PC2"],
            c=col, alpha=0.5, s=10, label=sp,
        )
    axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    axes[0].set_title("Species")
    axes[0].legend(markerscale=2, fontsize=9)
    axes[0].set_facecolor("none")

    # Right: colored by cluster
    unique_clusters = [c for c in coords_df["cluster"].dropna().unique() if c is not None]
    cmap = plt.cm.tab10
    for i, cl in enumerate(sorted(unique_clusters)):
        mask = coords_df["cluster"] == cl
        axes[1].scatter(
            coords_df.loc[mask, "PC1"],
            coords_df.loc[mask, "PC2"],
            c=[cmap(i / max(len(unique_clusters), 1))],
            alpha=0.5, s=10, label=cl,
        )
    axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    axes[1].set_title("Cluster")
    axes[1].legend(markerscale=2, fontsize=8, ncol=2)
    axes[1].set_facecolor("none")

    fig.suptitle("PCA of Z-scored Ephys Features (mouse + human)", fontsize=12)
    fig.patch.set_alpha(0)
    plt.tight_layout()

    if SAVE_FIGURES:
        fig.savefig(OUTDIR / "pca_ephys_cross_species.pdf", dpi=FIGURE_DPI, transparent=True)
        print(f"Saved: {OUTDIR / 'pca_ephys_cross_species.pdf'}")
    plt.show()
else:
    print("Not enough cells for PCA visualization.")

## 8. TODO: Tier 2 — ComBat Batch Correction

Parametric z-scoring (Tier 1) removes species-level mean/variance shifts per
feature, but does not account for correlated batch effects across features.
Tier 2 will apply ComBat (from `combat` or `neuroCombat` Python packages) to
further harmonize the feature space before cross-species comparison.

**TODO**:
1. Install neuroCombat: `pip install neuroCombat`
2. Apply ComBat with `batch = species` and `mod = cluster_label` as a
   covariate of interest (to protect biological variance)
3. Re-run permutation test and PCA on ComBat-corrected features
4. Compare Tier 1 vs Tier 2 cross-species distances

Reference data paths needed for Tier 2:
  - `results/ephys_convergence/combined_ephys_features.csv`
    (generated by `scripts/10_aggregate_ephys.py`)
  - `results/cross_species_rbh_mapping.csv`
    (generated by Module A/B cluster correspondence analysis)

In [ ]:
# Placeholder for ComBat import + call
# TODO: Uncomment and complete once neuroCombat is installed and data is available
#
# from neuroCombat import neuroCombat
#
# combat_data, _ = neuroCombat(
#     dat=features_zscored.T.values,
#     covars=pd.DataFrame({
#         "batch": combined_valid["species"].map({"mouse": 0, "human": 1}),
#         "cluster_label": combined_valid["cluster"].astype("category").cat.codes,
#     }),
#     batch_col="batch",
#     categorical_cols=["cluster_label"],
# )
# features_combat = pd.DataFrame(
#     combat_data.T,
#     columns=features_zscored.columns,
#     index=features_zscored.index,
# )

print("Tier 2 (ComBat) not yet implemented — see TODO above.")

## Summary

| Analysis | Result |
|----------|--------|
| Cells analyzed (mouse) | see above |
| Cells analyzed (human) | see above |
| Global permutation p-value | see above |
| CCKBC cross-species similarity | see above |

Next steps:
- Obtain cluster assignment files and re-run with real data
- Implement Tier 2 ComBat correction
- Compare CCKBC ephys profiles with published CCKBC markers